<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/08_lognormal_distributional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 8 — Distributional lognormal model

Notebook 7 combined varying intercepts and slopes with a lognormal likelihood. Its participant-level posterior predictive check reproduced each participant's level and rate of change, but left one systematic mismatch: all participants shared a single residual scale `sd_y`, which was too large for the steadiest participants and too small for the most variable ones.

This notebook lets the residual scale differ by participant as well. The mean structure is Notebook 7's, unchanged, and is supplied. The new piece, a hierarchical model for the participants' residual scales, is built here step by step: why it lives on the log scale, what its priors mean, and how to read the fitted scales.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

participants = sorted(sleep["Subject"].unique(), key=int)
participant_to_idx = {participant: i for i, participant in enumerate(participants)}
participant_idx = sleep["Subject"].map(participant_to_idx).to_numpy()

assert participant_idx.min() == 0
assert participant_idx.max() == len(participants) - 1
assert np.array_equal(
    np.asarray(participants)[participant_idx],
    sleep["Subject"].to_numpy(),
)

print(f"{len(participants)} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### Plotting helper

The participant plotting helper from the previous notebooks is supplied. It can optionally restrict plots to selected participants using the standard ArviZ `coords` argument. Panels appear in participant order (308, 309, …, 372), left to right and top to bottom.

In [ ]:
LM_VISUALS = {
    "pe_line": {"color": "C1"},
    "ci_band": {"color": "C0"},
    "observed_scatter": {"color": "black", "alpha": 1, "zorder": 3, "s": 10},
}

PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_participants(dt, group, var, coords=None):
    """One panel per participant, optionally restricted with ArviZ coords."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        group: reshape(dt[group].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
        "constant_data": reshape(dt["constant_data"].to_dataset()),
    })

    pc = azp.plot_lm(
        panels,
        x="days",
        y=var,
        y_obs="y",
        group=group,
        plot_dim="day",
        coords=coords,
        ci_prob=(0.50, 0.90),
        ci_kind="hdi",
        point_estimate="mean",
        smooth=False,
        cols=["participant"],
        col_wrap=6,
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={**LM_VISUALS, "xlabel": False, "ylabel": False},
    )

    pc.add_legend("prob", title="HDI")
    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    return pc

## 1. Build the residual-scale hierarchy

### 1.1 The model

The model keeps Notebook 7's mean structure and gives each participant their own residual scale:

$$
y_i \sim \operatorname{LogNormal}(\mu_{y,i}, sd_{y,s[i]})
$$

$$
\mu_{y,i}
=
b_{0,s[i]}
+
b_{1,s[i]}\,\mathrm{days}_i
$$

$$
b_{0,s}
\sim
\operatorname{Normal}(\mu_{b0}, sd_{b0})
$$

$$
b_{1,s}
\sim
\operatorname{Normal}(\mu_{b1}, sd_{b1})
$$

$$
\log sd_{y,s}
\sim
\operatorname{Normal}(\mu_{\log sd_y}, sd_{\log sd_y}).
$$

Here $s[i]$ identifies the participant who produced observation $i$. The second, third, and fourth lines are Notebook 7's mean structure, unchanged. What is new is the residual scale: in the first line it now carries the participant subscript $s[i]$, and the last line gives the participants' residual scales a population distribution of their own.

A model in which a parameter of the likelihood other than its location, here the residual scale, has its own sub-model is called a **distributional** model.

### 1.2 The supplied mean structure

The data, coordinates, and mean structure are Notebook 7's, with the same prior constants, and are supplied. The model is created here without a residual scale or a likelihood. As in Notebook 4, you will add the remaining nodes in separate `with model:` blocks.

In [ ]:
coords = {
    "obs_id": np.arange(len(sleep)),
    "participant": participants,
}

# Hyperprior constants for the intercept hierarchy (log reaction time), from Notebook 7
mu_mu_b0 = 5
sd_mu_b0 = 0.55
sd_sd_b0 = 1 / 6

# Hyperprior constants for the slope hierarchy (change in log reaction time per day),
# from Notebook 7
mu_mu_b1 = 0
sd_mu_b1 = 0.2
sd_sd_b1 = 0.05

with pm.Model(coords=coords) as model:
    days = pm.Data("days", sleep["Days"].to_numpy(), dims="obs_id")
    pidx = pm.Data("participant_idx", participant_idx, dims="obs_id")

    # Varying intercepts
    mu_b0 = pm.Normal("mu_b0", mu=mu_mu_b0, sigma=sd_mu_b0)
    sd_b0 = pm.Exponential("sd_b0", scale=sd_sd_b0)
    b0 = pm.Normal("b0", mu=mu_b0, sigma=sd_b0, dims="participant")

    # Varying slopes
    mu_b1 = pm.Normal("mu_b1", mu=mu_mu_b1, sigma=sd_mu_b1)
    sd_b1 = pm.Exponential("sd_b1", scale=sd_sd_b1)
    b1 = pm.Normal("b1", mu=mu_b1, sigma=sd_b1, dims="participant")

    # Location of log reaction time
    mu_y = pm.Deterministic(
        "mu_y",
        b0[pidx] + b1[pidx] * days,
        dims="obs_id",
    )

### 1.3 What does $sd_{y,s}$ describe?

How does it differ from `sd_y` in Notebook 7?

- answer here

### 1.4 Why does the model describe $\log sd_{y,s}$ rather than $sd_{y,s}$ itself?

Recall why Notebook 6 modeled log reaction time.

- answer here

### 1.5 What does $e^{\mu_{\log sd_y}}$ represent?

- answer here

### 1.6 Why estimate the residual scales hierarchically?

Each participant contributes eight reaction times, which must also determine that participant's intercept and slope. Why not estimate 18 separate residual scales, one participant at a time?

- answer here

### 1.7 What would a prior centered at zero imply?

On the log scale, a prior centered at zero can look like a neutral default. What typical residual scale does $\mu_{\log sd_y} = 0$ correspond to, and what would it mean for a participant's reaction times?

- answer here

### 1.8 What prior should we use for the typical residual scale?

Notebook 7 estimated a single residual scale shared by all participants, `sd_y` ≈ 0.08. Center the prior for $\mu_{\log sd_y}$ on the log of that value, and choose its standard deviation so that about 95% of the prior probability for the typical residual scale $e^{\mu_{\log sd_y}}$ lies between 0.03 and 0.22.

- answer here

### 1.9 Notebook 7's estimate came from these same data. Is it legitimate to use it here?

- answer here

### 1.10 What prior should we use for the between-participant variation in residual scale?

$sd_{\log sd_y}$ is the between-participant standard deviation of the log residual scales. Suppose we use an Exponential prior with mean 1/3. At that value, how many times larger is the residual scale of a participant one standard deviation above the typical participant? How different are two participants two standard deviations either side of the center, roughly the steadiest and the most variable in a group of 18?

- answer here

### 1.11 Store the prior constants for the residual-scale hierarchy.

Assign the values from Questions 1.8 and 1.10 to `mu_mu_log_sd_y`, `sd_mu_log_sd_y`, and `sd_sd_log_sd_y`, following the naming pattern of the mean-structure constants.

In [ ]:
# answer here

### 1.12 Add the population parameters of the residual-scale hierarchy.

In a `with model:` block, add the typical log residual scale `mu_log_sd_y` and the between-participant scale `sd_log_sd_y`, using the constants from Question 1.11.

In [ ]:
# answer here

### 1.13 Add each participant's residual scale.

Add `log_sd_y`, one value per participant drawn from the population distribution defined by `mu_log_sd_y` and `sd_log_sd_y`, in the same centered form as `b0` and `b1`. Then add the residual scales themselves as a `pm.Deterministic` named `sd_y`.

In [ ]:
# answer here

### 1.14 How does each observation get its participant's residual scale?

`sd_y` has one value per participant, but the likelihood needs one residual scale per observation.

- answer here

### 1.15 Complete the model.

Add the expected reaction time in milliseconds as a `pm.Deterministic` named `mean_rt`, now using each observation's participant-specific residual scale,

$$
\mathrm{E}[y_i] = \exp\!\left(\mu_{y,i} + \tfrac{1}{2}\,sd_{y,s[i]}^2\right),
$$

and the lognormal likelihood `y` for the observed reaction times.

In [ ]:
# answer here

### 1.16 Inspect the completed model.

In [ ]:
pm.model_to_graphviz(model)

### 1.17 Which nodes are new compared with Notebook 7's model?

Where does the residual-scale hierarchy enter the rest of the model?

- answer here

## 2. Check the prior implications

### 2.1 Criteria

The prior predictions should meet the criteria established in Notebook 1: predicted reaction times should not routinely be physically impossible, reaction times near baseline should mostly occupy a broadly plausible range, and the model should allow substantial change across the seven days without routinely generating absurd trajectories. The mean-structure priors are Notebook 7's and were examined there.

The new criterion concerns the residual-scale hierarchy: it should allow participants to differ noticeably in day-to-day variability, without making residual scales near 1 (Question 1.7), or enormous differences between participants, routine.

### 2.2 Draw from the prior.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=500,
        var_names=["mu_log_sd_y", "sd_log_sd_y", "y"],
        random_seed=RANDOM_SEED,
    )

### 2.3 Plot the priors of the two residual-scale hyperparameters.

Use `azp.plot_dist` for `mu_log_sd_y` and `sd_log_sd_y`, with 90% HDIs and the mean as the point estimate.

In [ ]:
# answer here

### 2.4 Are the residual-scale priors reasonable?

Translate each 90% HDI into residual scales, and into differences between participants.

- answer here

### 2.5 Plot the prior predictive reaction times.

In [ ]:
plot_participants(prior, "prior_predictive", "y")
plt.show()

### 2.6 Do these prior predictions meet the criteria established in Notebook 1?

Judge support, baseline scale, and changes across days, and compare with Notebook 7's prior predictive check.

- answer here

### 2.7 What causes the single-day spikes?

In two panels the mean line spikes on a single day: participant 351 (third row, first panel) on day 2, and participant 371 (third row, fifth panel) on day 7. Which part of the model can produce a spike confined to one day, and which prior must be responsible?

- answer here

### 2.8 Can these panels show whether the prior allows participants to differ in day-to-day variability?

- answer here

## 3. Fit and diagnose the model

### 3.1 Sample from the posterior.

This model is sampled with `target_accept=0.9` instead of the default 0.8. Question 3.2 asks why.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
    )

### 3.2 Why does this model need a higher `target_accept`?

With the default `target_accept` of 0.8, the sampler occasionally reports a divergence for this model, although Notebook 7's model sampled cleanly. A higher `target_accept` makes the sampler take smaller steps. Consider a steady participant such as 309: how precisely do its eight observations determine its intercept and slope when its residual scale is small, and when it is larger? Why could that make a single step size hard to choose?

- answer here

### 3.3 Check population-level diagnostics.

In [ ]:
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))

azs.summary(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "mu_log_sd_y", "sd_log_sd_y"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "mu_log_sd_y", "sd_log_sd_y"],
);

### 3.4 Do the population-level parameters meet the diagnostic criteria?

Use the criteria established in Notebook 1: no divergences, R-hat close to 1, adequate bulk and tail ESS, and well-mixed traces.

- answer here

### 3.5 Screen all participants, then inspect a representative subset.

In [ ]:
participant_diagnostics = azs.summary(
    idata,
    var_names=["b0", "b1", "log_sd_y"],
    kind="diagnostics",
    round_to=2,
)

display(pd.DataFrame(
    {
        "value": [
            participant_diagnostics["r_hat"].max(),
            participant_diagnostics["ess_bulk"].min(),
            participant_diagnostics["ess_tail"].min(),
        ]
    },
    index=["largest R-hat", "smallest bulk ESS", "smallest tail ESS"],
))

diagnostic_participants = [
    participants[0],
    participants[len(participants) // 2],
    participants[-1],
]
diagnostic_coords = {"participant": diagnostic_participants}

azs.summary(
    idata,
    var_names=["b0", "b1", "log_sd_y"],
    coords=diagnostic_coords,
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["b0", "b1", "log_sd_y"],
    coords=diagnostic_coords,
);

### 3.6 Do the participant-level parameters meet the same criteria?

- answer here

## 4. Examine the participant residual scales

### 4.1 Plot the posterior distributions of `mu_log_sd_y` and `sd_log_sd_y`.

In [ ]:
# answer here

### 4.2 What residual scale does a typical participant have?

Convert the 90% HDI for `mu_log_sd_y` from the summary in Question 3.3 into a residual scale, and compare it with Notebook 7's shared `sd_y`, about 0.08 (90% HDI roughly 0.07–0.09).

- answer here

### 4.3 How much do residual scales differ between participants?

Use the 90% HDI for `sd_log_sd_y`. By what factor does one standard deviation change a participant's residual scale?

- answer here

### 4.4 How far did the data move these two parameters from their priors?

Compare your plot in Question 4.1 with the prior plot in Question 2.3.

- answer here

### 4.5 Plot each participant's residual scale.

Use `azp.plot_forest` for `sd_y`, with 50% and 90% HDIs.

In [ ]:
# answer here

### 4.6 Does the fitted model support meaningful differences in residual scale between participants?

Base the answer on the participant intervals and the posterior for `sd_log_sd_y`, and express the difference between the steadiest and the most variable participants in milliseconds.

- answer here

## 5. Predictive consequences

### 5.1 Plot each participant's posterior expected reaction time.

Use `plot_participants` with the variable that gives the expected reaction time in milliseconds.

In [ ]:
# answer here

### 5.2 How do these expected trajectories compare with Notebook 7's?

Compare with the corresponding plot in Notebook 7 (its Question 5.1). Look separately at the mean lines and at the widths of the bands, which show uncertainty about each participant's expected reaction time.

- answer here

### 5.3 Generate and plot posterior predictive reaction times.

Generate replicated values of `y` from the fitted model, add them to `idata`, and compare them with the observations using `plot_participants`.

In [ ]:
# answer here

### 5.4 What is added when we move from `mean_rt` to posterior predictive `y`, and what is new about it in this model?

- answer here

### 5.5 Does the model reproduce the participant-level data, and does it resolve the mismatch Notebook 7 left?

Apply the posterior predictive criteria from Notebook 1: participants' overall levels, changes across days, and residual variation, emphasizing discrepancies that persist across a participant's observations. In Notebook 7's check, the steadiest participants' observations sat well inside bands that were too wide, and the most variable participants' observations fell outside bands that were too narrow.

- answer here

### 5.6 What would indicate adequate fit in an ECDF check?

The participant panels check conditional trajectories. Pooling all observations asks a different question: does the model reproduce the overall distribution of reaction times?

As in Notebooks 5–7, the observed ECDF should be broadly consistent with the posterior-predictive ECDFs across the whole distribution, without a persistent systematic displacement, particularly in the tails.

### 5.7 Plot the observed and posterior-predictive ECDFs.

Use `azp.plot_ppc_dist` with `kind="ecdf"`.

In [ ]:
# answer here

### 5.8 Does the model reproduce the overall distribution, and can this check distinguish it from Notebook 7's model?

- answer here

## 6. Summary

### 6.1 What has this notebook shown?

Summarize what letting the residual scale differ by participant showed about the model, the priors, sampling, the fitted residual scales, and the predictive checks.

- answer here